In [1]:
import pandas as pd
import threading
import time
import os

# Reconstrução das vistas

In [62]:
# Salvando uma cópia sem o último dia que está sendo simulado
df = pd.read_csv("vistas_lote/vistas_de_lote.csv")
df.to_csv("vistas_lote/vistas_de_lote copy.csv", index=False)

In [63]:
finaliza_laco = False
estado_stream = {}
lock_arquivo = threading.Lock()

metricas = ["media","contagem","maximo","minimo","mediana","desvio_padrao","variancia","moda"]

# Atualiza a vista rápida, calculando as métricas
def atualizar_vista_rapida(linha, header, var):
    global estado_stream
    
    # Seta o path da contagem incremental
    caminho = "vistas_tempo_real/contagem_incremental.csv"
    valores = [v.strip().replace('"', '') for v in linha.split(",")]

    try:
        # Pega o index da linha onde se encontra a coluna definida
        idx = header.index(var)
        valor_str = valores[idx].strip()
        
        # Tratamento de valores vazios ou nulos
        if not valor_str or valor_str == '' or valor_str == 'NA' or valor_str == 'null':
            return  # Pula esta linha se o valor for vazio
            
        valor = float(valor_str)
        timestamp_completo = valores[0].strip()
        
        # Verifica se o timestamp é válido
        if not timestamp_completo:
            return
            
    except (ValueError, IndexError, AttributeError) as e:
        # Pula silenciosamente linhas com erro
        return

    # Verifica se tem a linha, caso nao tenha, ele cria com valores padrões
    chave = var  # Usando apenas a variável como chave, sem data_dia
    if chave not in estado_stream:
        estado_stream[chave] = {"count": 0, "soma": 0, "max": valor, "min": valor, "valores": []}

    # Incrementa os estados
    estado = estado_stream[chave]
    estado["count"] += 1
    estado["soma"] += valor
    estado["max"] = max(estado["max"], valor)
    estado["min"] = min(estado["min"], valor)
    estado["valores"].append(valor)

    # Só calcula as métricas se tiver pelo menos 1 valor
    if estado["count"] == 0:
        return

    # Cálculo das métricas
    resultados = {
        "media": estado["soma"] / estado["count"],
        "contagem": estado["count"],
        "maximo": estado["max"],
        "minimo": estado["min"],
        "mediana": pd.Series(estado["valores"]).median(),
        "desvio_padrao": pd.Series(estado["valores"]).std(),
        "variancia": pd.Series(estado["valores"]).var(),
        "moda": pd.Series(estado["valores"]).mode()[0] if not pd.Series(estado["valores"]).mode().empty else None
    }
    
    # Trava a thread para que nenhuma outra thread altere/acesse simultaneamente
    with lock_arquivo:

        # Criando a nova linha pra atualizar o arquivo contagem incremental
        nova_linha = {
            "data": timestamp_completo,
            "var": var,
            **resultados
        }

        # Se o arquivo ainda não existe, cria com a linha
        if not os.path.exists(caminho):

            df = pd.DataFrame([nova_linha])

        else:

            # Se já existir, abre o df
            df = pd.read_csv(caminho)

            # Se estiver vazio
            if df.empty:

                df = pd.DataFrame([nova_linha])

            else:

                # Garantindo tipos corretos
                df["var"] = df["var"].astype(str)

                # Remove linha antiga da mesma variável
                df = df[
                    ~(
                        (df["var"] == str(var))
                    )
                ]

                # Adiciona nova linha atualizada
                df = pd.concat(
                    [df, pd.DataFrame([nova_linha])],
                    ignore_index=True
                )

        # Remove linhas completamente vazias ou com NaN
        df = df.dropna(how='all')
        
        # Remove linhas onde 'var' é vazia
        if 'var' in df.columns:
            df = df[df['var'].astype(str).str.strip() != '']
            df = df[df['var'].notna()]

        # Sobrescreve o CSV inteiro
        df.to_csv(caminho, index=False)

# Salva a ultima linha da contagem incremental na vista
# Salva a ultima linha da contagem incremental na vista
def reconstruir_vistas_lote():
    print("Iniciando a reconstrução da vista de lote")

    # Definindo arquivos com as informações novas e o vista de lote original
    entrada = "vistas_tempo_real/contagem_incremental.csv"
    saida = "vistas_lote/vistas_de_lote copy.csv"

    # Se não existir o arquivo de entrada não faz nada
    if not os.path.exists(entrada):
        print("Arquivo de entrada não encontrado")
        return

    df_ent = pd.read_csv(entrada)
    
    # Remove linhas vazias ou com var nula
    df_ent = df_ent.dropna(how='all')
    if 'var' in df_ent.columns:
        df_ent = df_ent[df_ent['var'].astype(str).str.strip() != '']
        df_ent = df_ent[df_ent['var'].notna()]

    # Se ele estiver vazio também não faz nada
    if df_ent.empty:
        print("Arquivo de entrada vazio após limpeza")
        return

    # Removendo as horas do campo data
    df_ent["data"] = df_ent["data"].astype(str).str.split(" ").str[0]

    # Arredondando as colunas para manter o padrão do arquivo original
    for col in ["media", "maximo", "minimo", "mediana", "desvio_padrao", "variancia", "moda"]:
        if col in df_ent.columns:
            df_ent[col] = pd.to_numeric(df_ent[col], errors='coerce').round(2)

    # Criar diretório se não existir
    os.makedirs(os.path.dirname(saida), exist_ok=True)

    # Se o arquivo de saída não existir só salva ele no lugar
    if not os.path.exists(saida):
        df_ent.to_csv(saida, index=False)
        print(f"Arquivo criado com {len(df_ent)} linhas")
    else:
        # Se o arquivo existir, abre, concatena e salva dnv
        df_lote = pd.read_csv(saida)
        
        # Concatena os dados (apenas adiciona, sem remover duplicatas)
        df_final = pd.concat([df_lote, df_ent], ignore_index=True)
        
        # NÃO remove duplicatas - mantém todo o histórico
        # df_final = df_final.drop_duplicates(subset=['var'], keep='last')
        
        # Remove apenas linhas completamente vazias
        df_final = df_final.dropna(how='all')
        
        # Salva o arquivo final mantendo todo o histórico
        df_final.to_csv(saida, index=False)
        print(f"Arquivo atualizado! Total de linhas: {len(df_final)}")

    print("Vista de lote atualizada com sucesso!")

    
def stream_dados(arq):
    global finaliza_laco
    while not finaliza_laco:
        linha = arq.readline().strip()
        if not linha:
            time.sleep(0.1)
            continue
        yield linha

def monitora_linhas(arquivo):
    # Enquanto o arquivo não existe ou ele não tem nada fica em stand by
    while not os.path.exists(arquivo) or os.path.getsize(arquivo) == 0:
        time.sleep(0.1)

    # Quando aparecer algo, chama a função de atualizar a vista rapida pra cada linha nova
    with open(arquivo, "r") as arq:
        header = [h.strip().replace('"', '') for h in arq.readline().strip().split(",")]
        for linha in stream_dados(arq):
            #for colum in ["duration_(secs)", "bytes", "age", "sales", "returned_amount"]:
            for colum in ["duration_(secs)","returned_amount"]:
                atualizar_vista_rapida(linha, header, colum)

def simular_stream_csv(entrada, saida, delay=0.05):
    global finaliza_laco
    
    os.makedirs(os.path.dirname(saida), exist_ok=True)
    
    with open(entrada, "r") as arq_in:
        with open(saida, "w") as arq_out:
            header = arq_in.readline()
            arq_out.write(header)
            arq_out.flush()
            for linha in arq_in:
                if finaliza_laco: break
                arq_out.write(linha)
                arq_out.flush()
                time.sleep(delay)

if __name__ == "__main__":
    
    # Seta os pahts de entrada, saida e stream
    arquivo_entrada = "dados_brutos/2017-03-21.csv"
    arquivo_stream = "dados_novos/fluxo.log"
    csv_tempo_real = "vistas_tempo_real/contagem_incremental.csv"

    os.makedirs("dados_novos", exist_ok=True)
    os.makedirs("vistas_tempo_real", exist_ok=True)

    if os.path.exists(csv_tempo_real): os.remove(csv_tempo_real)
    open(arquivo_stream, "w").close()

    # Define as threads
    t1 = threading.Thread(target=simular_stream_csv, args=(arquivo_entrada, arquivo_stream), daemon=True)
    t2 = threading.Thread(target=monitora_linhas, args=(arquivo_stream,), daemon=True)

    # Starta as threads
    t1.start()
    t2.start()

    try:
        while t1.is_alive():
            time.sleep(1)
    except KeyboardInterrupt:
        pass
    finally:
        # Finaliza as threads e salva a ultima linha da contagem incremental na vista de lote
        finaliza_laco = True
        t1.join()
        t2.join()
        reconstruir_vistas_lote()
        print("Processo finalizado.")

Iniciando a reconstrução da vista de lote
Arquivo atualizado! Total de linhas: 15
Vista de lote atualizada com sucesso!
Processo finalizado.
